# Stage 02 GCC to Relative Distance

这份 notebook 直接从 `gcc_phat + 3` 个麦间距离预测 `[r1, r2, r3]`，评估阶段再用三圆方程线性化最小二乘反解 `(x, y)`。

In [ ]:
from pathlib import Path
import json
import sys

try:
    import torch
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook needs an interpreter/kernel with torch installed. Please switch to your myenv Python before running it."
    ) from exc
from IPython.display import Image, display

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / '.gitignore').exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'python_impl').exists():
    raise RuntimeError(f'Failed to locate repo root from {NOTEBOOK_DIR}')

STAGE02_DIR = REPO_ROOT / 'python_impl' / 'experiments' / 'hypothesis_validation' / '02_source_localization_reflections'
SRC_DIR = STAGE02_DIR / 'src'
for path in (REPO_ROOT, SRC_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from gcc_reflection_notebook_utils import build_gcc_reflection_bundle, train_gcc_to_relative_distance_model

print('Repo root =', REPO_ROOT)
print('Stage02 dir =', STAGE02_DIR)

In [ ]:
H5_PATH = STAGE02_DIR / 'data' / 'source_localization_single_reflection_l1_stable_v3_w2.h5'
RESULT_DIR = STAGE02_DIR / 'results' / 'gcc_to_relative_distance_l1_stable_v3_w2'
ALPHA = 1.0
BETA = 1.0
LR = 1.0e-3
BATCH_SIZE = 128
EPOCHS = 40
SEED = 7
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('H5_PATH =', H5_PATH)
print('RESULT_DIR =', RESULT_DIR)
print('DEVICE =', DEVICE)

In [ ]:
bundle = build_gcc_reflection_bundle(H5_PATH)
print('geometry_filter_mode =', bundle.geometry_filter_mode)
print('min_triangle_area =', bundle.min_triangle_area)
print('max_jacobian_condition =', bundle.max_jacobian_condition)
print('max_triangle_angle_deg =', bundle.max_triangle_angle_deg)
print('near_ref_inside_threshold_m =', bundle.near_ref_inside_threshold_m)
print('source_region_rule_version =', bundle.source_region_rule_version)
print('rir_model =', bundle.rir_model)
print('air_attenuation_enabled =', bundle.air_attenuation_enabled)
print('air_attenuation_alpha_per_m =', bundle.air_attenuation_alpha_per_m)
print('reflection_profile_mix =', bundle.reflection_profile_mix)
print('profile_counts =', bundle.profile_counts)
print('split_sizes =', bundle.split_sizes)
print('removed train overlap =', bundle.train_overlap_removed)
print('gcc shape =', bundle.gcc_phat.shape)

In [ ]:
summary = train_gcc_to_relative_distance_model(
    bundle=bundle,
    result_dir=RESULT_DIR,
    alpha=ALPHA,
    beta=BETA,
    lr=LR,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    seed=SEED,
    device=DEVICE,
    live_plot=True,
)
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
for name in ['loss_curves.png', 'scatter_iid_test.png', 'scatter_geom_test.png']:
    path = RESULT_DIR / name
    print(f'\n{name}')
    display(Image(filename=str(path)))